<a href="https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I prioritize content for refresh when it is 91–180 days since its last update and still visible in search.

Content is considered in the target staleness range when days_since_last_update is between 91 and 180 days, inclusive. It is considered visible when impressions_90d >= 300.

The baseline score is higher for content with more recent search visibility within this target staleness range. The rule is intended for decision-support, not as a prediction of future performance.

### Reason codes

- `stale_visible`: content is 91–180 days old and has at least 300 impressions in the 90-day window.
- `no_priority_signal`: content does not satisfy the baseline priority condition.

In [ ]:
!git clone https://github.com/nihedzaoui/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [ ]:
!find /content/flyrank-ml-internship -type f | grep -E "content_refresh|\.csv$"


/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
DATA_PATH = Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

print("\nRequired columns:")
print(
    df[
        [
            "content_id",
            "client_id",
            "days_since_last_update",
            "impressions_90d",
        ]
    ].head()
)

# Basic signal checks
print("\nMissing values:")
print(
    df[
        ["days_since_last_update", "impressions_90d"]
    ].isna().sum()
)

print("\nDays since last update summary:")
print(df["days_since_last_update"].describe())

print("\n90-day impressions summary:")
print(df["impressions_90d"].describe())

Shape: (30000, 44)

Required columns:
             content_id          client_id  days_since_last_update  \
0  content_304f48230142  client_f369cb89fc                      20   
1  content_a1fb4e703a9e  client_4e07408562                      25   
2  content_9aa793d4d895  client_7f2253d7e2                      20   
3  content_331d6c4de07b  client_19581e27de                      22   
4  content_d99b7a2d90ca  client_3fdba35f04                      14   

   impressions_90d  
0             3803  
1            15320  
2            12581  
3            11751  
4            19140  

Missing values:
days_since_last_update    0
impressions_90d           0
dtype: int64

Days since last update summary:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

90-day impressions summary:
count     30000.000000
mean       5200.366300


### Signal checks

Before encoding the rule, I check two signals separately:

1. `days_since_last_update` — linked to refresh/staleness logic.
2. `impressions_90d` — linked to the volume/quick-win logic.

The checks are descriptive only. They do not use future outcomes or the declining label.

In [ ]:
# -----------------------------
# Signal 1: staleness
# -----------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30d",
        "31-90d",
        "91-180d",
        "181-365d",
        "365+d",
    ]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median"),
          mean_impressions_90d=("impressions_90d", "mean")
      )
      .reset_index()
)

print("Signal 1 — Staleness")
display(staleness_check)

Signal 1 — Staleness


,staleness_bucket,n,median_impressions_90d,mean_impressions_90d
0,0-30d,20480,470.0,4199.614062
1,31-90d,175,510.0,6506.748571
2,91-180d,9171,1692.0,7486.665140
3,181-365d,169,16.0,1206.893491
4,365+d,5,2.0,8.200000


In [ ]:
# -----------------------------
# Signal 2: search volume
# -----------------------------

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 299, 999, 2999, 9999, np.inf],
    labels=[
        "<100",
        "100-299",
        "300-999",
        "1k-2.9k",
        "3k-9.9k",
        "10k+",
    ]
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_staleness_days=("days_since_last_update", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("Signal 2 — Search volume")
display(volume_check)

Signal 2 — Search volume


,volume_bucket,n,median_staleness_days,median_ctr
0,<100,7994,20.0,0.00
1,100-299,3254,22.0,0.00
2,300-999,5240,22.0,0.10
3,1k-2.9k,5229,22.0,0.14
4,3k-9.9k,4681,22.0,0.20
5,10k+,3602,25.0,0.23


### Signal verdicts

**Staleness — MIXED.**
The bucket table does not show a monotonic relationship between age and search visibility. Content aged 91–180 days has the highest median impressions (1,692), while content older than 180 days has much lower median impressions (16 for 181–365 days and 2 for 365+ days). The oldest buckets are also small (`n=169` and `n=5`), so the apparent decline should be treated cautiously.

**Search volume — CONFIRMED.**
The bucket table shows a clear directional relationship between 90-day impressions and median CTR. Median CTR increases from 0.00 in the lowest-volume buckets to 0.23 in the 10k+ bucket. The pattern is consistent across relatively large bucket sizes, so search volume is a useful descriptive signal for prioritization, although this does not establish causality.

### Rule decision

The staleness signal is mixed rather than monotonic. Content in the 91–180 day bucket has substantially higher median search visibility than content older than 180 days.

Therefore, this baseline focuses on the 91–180 day range rather than assuming that older content is always a stronger refresh candidate.

The rule requires both:
- 91–180 days since the last update
- at least 300 impressions in the last 90 days

The rule is a transparent decision-support baseline, not a prediction of future performance.

## 2. Build the ranked queue (writes the CSV)

### Baseline rule

I prioritize content that is 91–180 days since its last update and still has at least 300 impressions in the last 90 days.

The score is the observed `impressions_90d` value for items satisfying both conditions. Items outside this condition receive a score of zero.

**Reason code:** `stale_visible`

**Action label:** `review_for_refresh`

This is a hand-written rule with no fitted model and no label-derived inputs. The score is used only to rank the review queue.


In [ ]:
df["is_target_staleness"] = (
    (df["days_since_last_update"] >= 91)
    & (df["days_since_last_update"] <= 180)
)

df["has_meaningful_volume"] = (
    df["impressions_90d"] >= 300
)

df["score"] = np.where(
    df["is_target_staleness"] & df["has_meaningful_volume"],
    df["impressions_90d"],
    0
)

df["reason_code"] = np.where(
    df["score"] > 0,
    "stale_visible",
    "no_priority_signal"
)

df["action"] = np.where(
    df["score"] > 0,
    "review_for_refresh",
    "no_action"
)

df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

queue = df[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position",
    ]
].copy()

OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Rows: {len(queue):,}")
print(f"Priority candidates: {(queue['score'] > 0).sum():,}")

display(queue.head(20))



Saved to: work/outputs/baseline_action_score.csv
Rows: 30,000
Priority candidates: 7,212


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position
0,1,content_5fe46e04994d,client_4e07408562,517715,stale_visible,review_for_refresh,104,517715,0.14,4.2
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,stale_visible,review_for_refresh,104,443434,0.21,27.9
2,3,content_2c2606c5d176,client_19581e27de,347399,stale_visible,review_for_refresh,104,347399,0.53,4.2
3,4,content_cb112fce36be,client_19581e27de,309910,stale_visible,review_for_refresh,104,309910,0.16,5.6
4,5,content_9532f197bbc8,client_4e07408562,309192,stale_visible,review_for_refresh,104,309192,0.87,2.0
5,6,content_36ff89c8214e,client_19581e27de,295097,stale_visible,review_for_refresh,104,295097,0.05,7.3
6,7,content_b28d1efd668f,client_6208ef0f77,286608,stale_visible,review_for_refresh,104,286608,0.06,26.2
7,8,content_813e88069237,client_6208ef0f77,233561,stale_visible,review_for_refresh,104,233561,0.06,26.2
8,9,content_c21024970297,client_19581e27de,211366,stale_visible,review_for_refresh,104,211366,0.41,5.1
9,10,content_c8e9d6ab9013,client_19581e27de,208678,stale_visible,review_for_refresh,104,208678,0.00,9.7


In [ ]:
# Write the ranked queue
OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nSaved to: {OUTPUT_PATH}")
print(f"Rows written: {len(queue):,}")


Saved to: work/outputs/baseline_action_score.csv
Rows written: 30,000


## 3. Top-20 review

The top 20 items are reviewed as decision-support candidates rather than guaranteed actions.

Each item receives an action, a reason code, a confidence note, and a condition that could make the recommendation wrong.

The baseline is intentionally simple: high impressions indicate observed search visibility, but they do not prove that refreshing the content will improve future performance.

In [ ]:
# ============================================
# 3. Top-20 review
# ============================================

# Select the top 20 ranked items
top20 = queue.head(20).copy()


# Confidence note for each recommendation
def confidence_note(row):
    if row["reason_code"] == "stale_visible":
        return (
            "Moderate: the item is 91–180 days old and has at least "
            "300 impressions in the 90-day window."
        )

    return (
        "Low: the item does not satisfy the baseline priority condition."
    )


# Explain what could make the recommendation wrong
def what_would_make_it_wrong(row):
    if row["reason_code"] == "stale_visible":
        return (
            "The recommendation could be wrong if the content is intentionally "
            "stable, the recorded update date is incomplete, or high impressions "
            "do not represent a useful refresh opportunity."
        )

    return (
        "The item may still deserve attention for a reason not captured "
        "by this simple baseline."
    )


# Add the review columns
top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)


# Select the columns required for the review
review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
].copy()


# Display the Top-20 review
display(review)

,rank,content_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,review_for_refresh,stale_visible,517715,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
1,2,content_2dba2b1f9536,review_for_refresh,stale_visible,443434,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
2,3,content_2c2606c5d176,review_for_refresh,stale_visible,347399,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
3,4,content_cb112fce36be,review_for_refresh,stale_visible,309910,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
4,5,content_9532f197bbc8,review_for_refresh,stale_visible,309192,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
5,6,content_36ff89c8214e,review_for_refresh,stale_visible,295097,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
6,7,content_b28d1efd668f,review_for_refresh,stale_visible,286608,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
7,8,content_813e88069237,review_for_refresh,stale_visible,233561,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
8,9,content_c21024970297,review_for_refresh,stale_visible,211366,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...
9,10,content_c8e9d6ab9013,review_for_refresh,stale_visible,208678,Moderate: the item is 91–180 days old and has ...,The recommendation could be wrong if the conte...


## 4. Weak picks + leakage check
### Weak picks

The baseline can produce weak picks because it only considers staleness and search visibility.

A high score means that an item satisfies the rule and has observed search visibility. It does not mean that refreshing the item will necessarily improve future performance.

Potential weak picks include content that is intentionally stable, content whose update date is incomplete, or content with high impressions but limited practical refresh opportunity.

### Leakage check

The baseline uses only `days_since_last_update` and `impressions_90d`.

It does not use future trend variables or the declining label.

The following fields are explicitly excluded from the rule:

* `trend_pct`
* `trend_direction`
* `is_declining_label`


In [ ]:
# ============================================
# 4. Weak picks + leakage check
# ============================================

print("Weak-pick candidates:")

weak_picks = queue[
    queue["reason_code"] == "stale_visible"
].tail(5)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "days_since_last_update",
            "impressions_90d",
            "ctr",
            "avg_position",
        ]
    ]
)


# ============================================
# Leakage check
# ============================================

baseline_inputs = [
    "days_since_last_update",
    "impressions_90d",
]

forbidden_inputs = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
]

print("\nBaseline inputs:")
for col in baseline_inputs:
    print(f"  - {col}")

print("\nLeakage-sensitive columns:")
for col in forbidden_inputs:
    print(f"  - {col}")


assert not any(
    col in baseline_inputs
    for col in forbidden_inputs
)

print("\nLeakage check: PASS")
print("No label-derived or future trend variables are used.")


Weak-pick candidates:


,rank,content_id,score,reason_code,days_since_last_update,impressions_90d,ctr,avg_position
7207,7208,content_4e2aaf9add7e,301,stale_visible,104,301,0.00,64.4
7208,7209,content_c9f5d16ec182,301,stale_visible,104,301,0.33,21.9
7209,7210,content_664c76db7888,301,stale_visible,104,301,1.33,22.1
7210,7211,content_53459e8ddd7f,300,stale_visible,104,300,0.67,13.9
7211,7212,content_1eebdcbead2d,300,stale_visible,104,300,0.00,20.0



Baseline inputs:
  - days_since_last_update
  - impressions_90d

Leakage-sensitive columns:
  - trend_pct
  - trend_direction
  - is_declining_label

Leakage check: PASS
No label-derived or future trend variables are used.


### Self-check

* [x] Two signal checks were completed with visible bucket tables and `n`.
* [x] Staleness was judged MIXED based on the observed bucket pattern.
* [x] Search volume was judged CONFIRMED based on its directional relationship with CTR.
* [x] One transparent rule was encoded.
* [x] The rule uses the 91–180 day staleness range and 300+ impressions.
* [x] The rule produces one score, one reason code, and one action label.
* [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
* [x] The top 20 are reviewed with action, reason code, confidence, and failure conditions.
* [x] Weak picks are discussed.
* [x] No future-window variables are used.
* [x] No label-derived variables are used.
* [x] No client names, URLs, or private queries are included.
* [x] The notebook is intended to run from top to bottom without manual intervention.
